# Causal Forest — cấu hình `rare-outcome` trên split Sprint 2/3

Protocol: `configs/causal_forest_rare_outcome_protocol_v1.json` — **đăng ký trước khi chạy**.

## Notebook này khác lần chạy trước ở hai điểm

**1. Cấu hình sửa đúng ràng buộc đang bó.** Lần trước dùng `min_samples_leaf=500`. Với
treatment 85/15 và conversion rate control `0,1938%`, mỗi lá chỉ có kỳ vọng **0,145 sự kiện
control**; sau honest splitting còn khoảng 0,073. Tức đại đa số lá có nhánh control rỗng, và
Causal Forest đang lấy hiệu số treated − control với một vế gần như không có thông tin.

`min_samples_leaf=10000` nâng con số đó lên khoảng **2,9**.

Lưu ý: profile `research` có sẵn trong repo **không** phải cải tiến cho bài toán này — nó dùng
`min_samples_leaf=200`, tức đi sai hướng trên đúng ràng buộc đó.

**2. Chạy trên split Sprint 2/3, không phải holdout Sprint 1.** Fit trên development
(5.591.836 dòng), predict trên confirmation (1.397.959 dòng). Nhờ vậy điểm số đặt chung được
bảng confirmation Sprint 3, chấm bằng **đúng DR signal đã đóng băng** — chênh lệch giữa hai
model không lẫn với chênh lệch giữa hai tín hiệu chấm điểm.

## Chỉ một stage

Không có thang 20 → 30 → 50% như lần trước. Development pool là một tập cố định, nên đây là
**một lần chạy duy nhất**.

## Notebook này không làm gì

Nó **không** chấm điểm. Kaggle chỉ fit và xuất điểm CATE; việc chấm chạy ở local bằng
`scripts/evaluate_causal_forest.py`, vì nó cần `output/sprint3/confirmation_predictions.npz`
— file bị `.gitignore` loại nên không có trong bản clone.

## Kỳ vọng kết quả

Nên chuẩn bị tinh thần **CI chứa 0**. Confirmation có 1.397.959 dòng, *nhỏ hơn* holdout
Sprint 1 (2.096.940), nên độ phân giải còn thấp hơn lần đo trước.

Giá trị của lần chạy này là đóng lỗ hổng lập luận "Causal Forest thua vì lá quá nhỏ cho outcome
hiếm" — một phản bác hiện tại **đúng**. Nó biến kết luận hoà thành kết luận vững, chứ không
nhằm đảo chiều kết quả.

## Cell 1 — Đọc tài nguyên thật của session

In [1]:
import os, platform, shutil
import psutil

vm = psutil.virtual_memory()
disk = shutil.disk_usage('/kaggle/working')
print(f'python            {platform.python_version()}')
print(f'logical cpus      {os.cpu_count()}')
print(f'physical cpus     {psutil.cpu_count(logical=False)}')
print(f'RAM total         {vm.total / 2**30:.2f} GB')
print(f'RAM available     {vm.available / 2**30:.2f} GB')
print(f'working disk free {disk.free / 2**30:.2f} GB')
print()

found = False
for root, _, files in os.walk('/kaggle/input'):
    for name in files:
        p = os.path.join(root, name)
        print(f'{os.path.getsize(p):>16,} byte  {p}')
        found = True
if not found:
    print('KHONG CO FILE NAO trong /kaggle/input')
    print('-> Add Input o thanh ben phai, bam nut + ben canh ten dataset.')

if vm.total / 2**30 < 20:
    print()
    print('CANH BAO: RAM total duoi 20 GB.')
    print('Run nay fit tren 5.591.836 dong - lon hon lan chay truoc (4.892.857).')
    print('Gate se tu dung neu peak vuot 75% RAM.')

python            3.12.13
logical cpus      4
physical cpus     2
RAM total         31.35 GB
RAM available     30.36 GB
working disk free 19.50 GB

   3,248,115,221 byte  /kaggle/input/datasets/datthnh/criteo-uplift-v21/criteo-uplift-v2.1.csv


## Cell 2 — Xác minh checksum dữ liệu

In [2]:
import glob, hashlib, os

# Hai dang byte hop le cua cung mot du lieu:
#   csv.gz    311.422.618 byte  - ban tai ve goc
#   csv     3.248.115.221 byte  - ban Kaggle da giai nen khi upload
SHA_GZ  = '2716e1bf0fd157a93b5bf86924d9088419dfbac2022c6cd90030220634f616dc'
SHA_CSV = 'e4d7c710ca1f38e523309d0f8a0745d1b53e7392d51f20d1088b6cfeaef222ef'

matches = sorted(glob.glob('/kaggle/input/**/*criteo*uplift*v2.1.csv*', recursive=True))
assert matches, (
    'Khong tim thay file Criteo. Kiem tra hai thu:\n'
    '  1. Da bam nut + de attach dataset chua (Add Input o thanh ben phai)?\n'
    '  2. Ten file co chua "criteo", "uplift" va "v2.1" khong? Xem output Cell 1.'
)
DATA_PATH = matches[0]
if len(matches) > 1:
    print('CANH BAO: tim thay nhieu file, dung file dau tien.')
    for m in matches:
        print('   ', m)

expected = SHA_GZ if DATA_PATH.endswith('.gz') else SHA_CSV
digest = hashlib.sha256()
with open(DATA_PATH, 'rb') as handle:
    for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
        digest.update(chunk)
actual = digest.hexdigest()

print('path  ', DATA_PATH)
print('size  ', f'{os.path.getsize(DATA_PATH):,} byte')
print('dang  ', 'csv.gz' if DATA_PATH.endswith('.gz') else 'csv (Kaggle da giai nen)')
print('sha256', actual)
assert actual == expected, (
    f'Checksum sai.\n  thuc te : {actual}\n  mong doi: {expected}\n'
    'Du lieu khong phai Criteo v2.1 nguyen ven.'
)
print()
print('checksum OK')

path   /kaggle/input/datasets/datthnh/criteo-uplift-v21/criteo-uplift-v2.1.csv
size   3,248,115,221 byte
dang   csv (Kaggle da giai nen)
sha256 e4d7c710ca1f38e523309d0f8a0745d1b53e7392d51f20d1088b6cfeaef222ef

checksum OK


## Cell 3 — Đưa repository vào session

In [3]:
import os, shutil, subprocess

USE_GITHUB = True
REPO_URL = 'https://github.com/ThanhDatVN/Causal-Uplift-for-Activation-and-Retention.git'
REPO_DATASET = '/kaggle/input/<ten-dataset-repo>'   # chi dung khi USE_GITHUB = False
REPO_DIR = '/kaggle/working/repo'

if not os.path.exists(REPO_DIR):
    if USE_GITHUB:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    else:
        assert os.path.isdir(REPO_DATASET), f'Khong thay {REPO_DATASET}'
        shutil.copytree(REPO_DATASET, REPO_DIR, dirs_exist_ok=True)

os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

for required in ('scripts', 'src', 'configs'):
    assert os.path.isdir(required), f'Thieu thu muc {required} trong repo'
for required in (
    'scripts/kaggle_causal_forest_gate.py',
    'scripts/train_causal_forest.py',
    'configs/causal_forest_rare_outcome_protocol_v1.json',
):
    assert os.path.isfile(required), f'Thieu file {required}'
print('repo OK')

Cloning into '/kaggle/working/repo'...


cwd: /kaggle/working/repo
repo OK


## Cell 4 — Cài dependency có ghim phiên bản

In [4]:
import subprocess, sys

packages = [
    'econml==0.16.0',
    'scikit-learn>=1.4,<1.7',   # rang buoc cung cua econml 0.16
    'shap>=0.38.1,<0.49.0',     # rang buoc cung cua econml 0.16
    'lightgbm>=4.5',
    'psutil',
]
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + packages,
    capture_output=True, text=True,
)
print(result.stdout[-2000:])
print(result.stderr[-2000:])
print('exit code:', result.returncode)
assert result.returncode == 0, 'pip install that bai, doc stderr o tren'
print()
print('Cai xong. KHONG can restart kernel - moi lenh nang deu chay trong subprocess.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.3/155.3 kB 6.1 MB/s eta 0:00:00


exit code: 0

Cai xong. KHONG can restart kernel - moi lenh nang deu chay trong subprocess.


## Cell 5 — Vì sao **không** cần restart kernel

Kernel hiện tại đã nạp sẵn `scikit-learn` bản cũ trước khi `pip install` chạy, nên nếu import
`econml` ngay trong kernel này thì có thể dính bản đã nạp trong bộ nhớ.

Notebook tránh hẳn vấn đề đó: **mọi bước nặng đều chạy trong `subprocess`**, tức một tiến trình
Python mới đọc lại `site-packages` từ đĩa. Cell 6 xác minh điều này bằng cách in phiên bản *bên
trong* một subprocess chứ không phải trong kernel.

Đây cũng là lý do notebook chạy được nguyên trạng bằng `Save & Run All`.

## Cell 6 — Kiểm tra phiên bản trong tiến trình con

In [5]:
import glob, os, subprocess, sys, textwrap

REPO_DIR = '/kaggle/working/repo'
OUTPUT_ROOT = '/kaggle/working/output/causal_forest'
os.chdir(REPO_DIR)

matches = sorted(glob.glob('/kaggle/input/**/*criteo*uplift*v2.1.csv*', recursive=True))
assert matches, 'Khong tim thay du lieu Criteo. Xem lai Cell 1 va Cell 2.'
DATA_PATH = matches[0]

print('cwd      ', os.getcwd())
print('data     ', DATA_PATH)
print('output   ', OUTPUT_ROOT)
print()

check = textwrap.dedent('''
    import inspect
    import numpy, scipy, sklearn, lightgbm, econml, pandas
    for m in (numpy, scipy, sklearn, lightgbm, econml, pandas):
        print(f'{m.__name__:14s} {m.__version__}')
    from packaging.version import Version
    assert Version(sklearn.__version__) < Version('1.7'), (
        f'scikit-learn {sklearn.__version__} qua moi cho econml 0.16')
    assert econml.__version__ == '0.16.0', f'econml {econml.__version__}, can 0.16.0'
    from econml.dml import CausalForestDML
    assert 'inference' in inspect.signature(CausalForestDML.__init__).parameters
    print('OK')
''')
result = subprocess.run([sys.executable, '-c', check], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])
assert result.returncode == 0, 'Moi truong subprocess chua dung. Chay lai Cell 4.'
print('moi thu OK')

cwd       /kaggle/working/repo
data      /kaggle/input/datasets/datthnh/criteo-uplift-v21/criteo-uplift-v2.1.csv
output    /kaggle/working/output/causal_forest

numpy          2.0.2
scipy          1.16.3
sklearn        1.6.1
lightgbm       4.6.0
econml         0.16.0
pandas         2.3.3
OK

moi thu OK


## Cell 7 — Đọc protocol đã đăng ký

In ra cấu hình sẽ chạy **trước khi** chạy. Nếu con số ở đây khác điều bạn định làm thì dừng lại
và sửa protocol, không sửa tham số dòng lệnh — quy tắc của dự án là cấu hình phải được đăng ký
trước, và kết quả báo cáo là một run mới chứ không thay dòng Causal Forest cũ.

In [6]:
import json

with open('configs/causal_forest_rare_outcome_protocol_v1.json') as handle:
    protocol = json.load(handle)

print('protocol_id :', protocol['protocol_id'])
print('status      :', protocol['status'])
print()
print('--- cau hinh ---')
for key, value in protocol['configuration'].items():
    print(f'  {key:20s} {value}')
print()
print('--- du lieu ---')
for key, value in protocol['data'].items():
    print(f'  {key:22s} {value}')
print()
print('--- ky vong ket qua ---')
print(' ', protocol['decision_rule']['expected_outcome'])

# Ky vong su kien control moi la, tinh thang tu ty le da biet.
leaf = protocol['configuration']['min_samples_leaf']
control_share, control_rate = 0.15, 0.001938
print()
print(f'min_samples_leaf={leaf} -> ky vong '
      f'{leaf * control_share * control_rate:.3f} su kien control moi la')
print(f'(cau hinh cu min_samples_leaf=500 -> '
      f'{500 * control_share * control_rate:.3f})')

protocol_id : causal-forest-rare-outcome-v1
status      : completed_not_promoted

--- cau hinh ---
  profile              rare-outcome
  n_estimators         500
  min_samples_leaf     10000
  cv                   3
  max_samples          0.45
  honest               True
  inference            False
  model_y              LGBMRegressor(n_estimators=200, max_depth=5)
  model_t              DummyClassifier(strategy='prior')
  discrete_treatment   True
  seed                 42

--- du lieu ---
  split                  sprint3
  train_on               development = fit + validation cua Sprint 2
  train_rows             5591836
  predict_on             confirmation cua Sprint 2
  predict_rows           1397959
  alignment_contract     source_index phai trung khit tung phan tu voi output/sprint3/confirmation_predictions.npz
  split_hash_guard       sha256 cua fit/validation/confirmation source-index phai khop SPRINT2_SPLIT_HASHES

--- ky vong ket qua ---
  CI chua 0. Confirmation co 1.397.9

## Cell 8 — Smoke code path (khoảng 2 phút)

Chạy đúng đường code với một mẫu train rất nhỏ, để bắt lỗi cấu hình **trước** khi bỏ ra một
lần chạy dài. Cell này ghi vào thư mục riêng nên không đụng artifact thật.

`--train-subsample` được ghi vào artifact, nên một lần smoke không thể bị nhầm thành run thật.

In [7]:
import subprocess, sys, time

started = time.time()
smoke = subprocess.run(
    [sys.executable, 'scripts/train_causal_forest.py',
     '--data-path', DATA_PATH,
     '--split', 'sprint3',
     '--profile', 'rare-outcome',
     '--train-subsample', '0.02',
     '--n-estimators', '8',
     '--cv', '2',
     '--output-dir', '/kaggle/working/output/cf_smoke'],
    capture_output=True, text=True,
)
print(smoke.stdout[-3000:])
if smoke.returncode != 0:
    print('STDERR:', smoke.stderr[-3000:])
assert smoke.returncode == 0, 'Smoke that bai - dung lai, dung chay Cell 9.'
print()
print(f'smoke OK trong {time.time() - started:.0f}s. Split hash va contract deu dat.')

[smoke] --train-subsample=0.02: train 5,591,836 -> 111,837. KHONG phai run that.
[split] mode=sprint3 frac=0.5 train=111,837 test=1,397,959 time=48.5s
[leaf] min_samples_leaf=10000 -> ky vong 2.861 su kien control moi la (control_share=0.1500 control_rate=0.001908)
[fit] profile=rare-outcome n_estimators=8 min_samples_leaf=10000 cv=2 max_samples=0.45 inference=False
[fit] done fit_time=1.9s
[holdout] y_sha256=b0d9e82e56168c4da09ef0d1 t_sha256=a3d65107174cffa80663de3d
[write] /kaggle/working/output/cf_smoke/cate_causal_forest_rare_outcome.npy n=1,397,959 mean=0.000937 total_time=56.7s


smoke OK trong 61s. Split hash va contract deu dat.


## Cell 9 — Chạy thật

Fit trên 5.591.836 dòng, predict trên 1.397.959 dòng.

### Đo thật từ lần chạy 13/08/2026

| | |
|---|---|
| Wall time | **107,4 phút** |
| Peak RSS | **28,46 GB** = 90,8% của 31,35 GB |
| Gate 75% | **fail** |
| Artifact | 1.397.959 dòng, `all finite: True`, `aligned: True` |

Fit **chạy xong và điểm số hợp lệ**; gate fail thuần tuý vì ngưỡng RAM.

Bộ nhớ bị chi phối bởi `n_estimators × max_samples × n_rows` — tức số subsample được giữ cho
từng cây — chứ không phải bởi độ sâu cây. So với cấu hình cũ, tích đó tăng khoảng `5,1` lần
(`500/200 × 0,45/0,25 × 5,59/4,89`), nên `min_samples_leaf` lớn hơn làm cây nông đi vẫn không
bù lại được.

### Giảm RAM mà **không** đổi kết quả

`--n-jobs` là tham số song song, không phải tham số model. Đã kiểm chứng: fit hai lần với
`n_jobs=1` và `n_jobs=2` cho điểm số **giống hệt từng bit**. Hạ nó xuống là thao tác vận hành
hợp lệ, không phải tinh chỉnh sau khi nhìn kết quả.

Đặt `N_JOBS = 2` ở cell dưới nếu muốn qua gate; đổi lại là thời gian dài hơn.

Gate vẫn đo và ghi lại mọi thứ. Nếu nó fail, đó là **kết quả cần báo cáo**, không phải lỗi cần
lách — và Cell 10 vẫn đóng gói artifact bình thường để không mất công fit.

In [8]:
import json, subprocess, sys, time

# 4 = mac dinh (-1, dung het CPU). Dat 2 de giam peak RAM; ket qua khong doi.
N_JOBS = -1

started = time.time()
command = [sys.executable, 'scripts/kaggle_causal_forest_gate.py',
           '--data-path', DATA_PATH,
           '--frac', '0.5',
           '--split', 'sprint3',
           '--profile', 'rare-outcome',
           '--output-root', OUTPUT_ROOT,
           '--max-ram-fraction', '0.75']
if N_JOBS != -1:
    command += ['--n-jobs', str(N_JOBS)]

result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout[-5000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-4000:])

path = f'{OUTPUT_ROOT}/sprint3_rare_outcome/gate_manifest.json'
with open(path) as handle:
    manifest = json.load(handle)
runtime = manifest['runtime']
contract = manifest['artifact_contract']
print()
print('status               ', manifest['status'])
print('split                ', manifest.get('split'))
print('profile              ', manifest.get('profile'))
print(f"peak RSS              {runtime['peak_process_tree_rss_gb']:.2f} GB")
print(f"peak RAM fraction     {runtime['peak_process_tree_ram_fraction']:.3f}")
print(f"wall time             {runtime['wall_seconds'] / 60:.1f} phut")
print('score rows           ', contract['score_rows'])
print('all finite           ', contract['all_finite'])
print('aligned              ', contract['aligned'])
print()
print(f'tong thoi gian cell: {(time.time() - started) / 60:.1f} phut')

# KHONG assert o day. Gate co the fail vi nguong RAM trong khi diem so van hop le;
# dung lai ngay se lam mat artifact vua fit xong. Cell 10 dong goi truoc, Cell 11
# moi ra ket luan.
if manifest['status'] != 'passed':
    print()
    print('GATE FAIL - nhung van chay tiep de dong goi artifact.')
    print('Doc ket luan o Cell 11.')

[data] /kaggle/input/datasets/datthnh/criteo-uplift-v21/criteo-uplift-v2.1.csv dạng csv, checksum khớp
{
  "created_utc": "2026-08-17T04:37:27.183498+00:00",
  "status": "failed",
  "scope": "resource_and_artifact_integrity_gate_only",
  "fraction": 0.5,
  "profile": "rare-outcome",
  "split": "sprint3",
  "command": [
    "/usr/bin/python3",
    "/kaggle/working/repo/scripts/train_causal_forest.py",
    "--data-path",
    "/kaggle/input/datasets/datthnh/criteo-uplift-v21/criteo-uplift-v2.1.csv",
    "--frac",
    "0.5",
    "--split",
    "sprint3",
    "--profile",
    "rare-outcome",
    "--output-dir",
    "/kaggle/working/output/causal_forest/sprint3_rare_outcome",
    "--seed",
    "42",
    "--n-jobs",
    "-1"
  ],
  "data": {
    "path": "/kaggle/input/datasets/datthnh/criteo-uplift-v21/criteo-uplift-v2.1.csv",
    "sha256": "e4d7c710ca1f38e523309d0f8a0745d1b53e7392d51f20d1088b6cfeaef222ef",
    "form": "csv"
  },
  "runtime": {
    "logical_cpus": 4,
    "physical_cpus": 2,
 

## Cell 10 — Đóng gói output để tải về

In [9]:
import json, os, pathlib, shutil

archive = shutil.make_archive(
    '/kaggle/working/causal_forest_rare_outcome', 'zip', OUTPUT_ROOT,
)
print(f'{archive}  {os.path.getsize(archive) / 2**20:.1f} MB')
print()
for root, _, files in os.walk(OUTPUT_ROOT):
    for name in sorted(files):
        p = os.path.join(root, name)
        print(f'{os.path.getsize(p) / 2**20:9.2f} MB  {p}')

# In manifest ra ngay, de phong khong tai duoc file.
path = pathlib.Path(OUTPUT_ROOT) / 'sprint3_rare_outcome' / 'gate_manifest.json'
print()
print('=' * 30, 'gate_manifest', '=' * 30)
print(json.dumps(json.loads(path.read_text()), indent=2))

try:
    from IPython.display import FileLink, display
    print()
    print('Bam link duoi day de tai zip (chi dung khi chay interactive):')
    display(FileLink('causal_forest_rare_outcome.zip'))
except Exception as exc:
    print('Khong tao duoc FileLink:', exc)

/kaggle/working/causal_forest_rare_outcome.zip  15.2 MB

    10.67 MB  /kaggle/working/output/causal_forest/sprint3_rare_outcome/cate_causal_forest_rare_outcome.npy
     0.00 MB  /kaggle/working/output/causal_forest/sprint3_rare_outcome/gate_manifest.json
    21.34 MB  /kaggle/working/output/causal_forest/sprint3_rare_outcome/holdout_test_yt.npz
     0.00 MB  /kaggle/working/output/causal_forest/sprint3_rare_outcome/train.log

============================== gate_manifest ==============================
{
  "created_utc": "2026-08-17T04:37:27.183498+00:00",
  "status": "failed",
  "scope": "resource_and_artifact_integrity_gate_only",
  "fraction": 0.5,
  "profile": "rare-outcome",
  "split": "sprint3",
  "command": [
    "/usr/bin/python3",
    "/kaggle/working/repo/scripts/train_causal_forest.py",
    "--data-path",
    "/kaggle/input/datasets/datthnh/criteo-uplift-v21/criteo-uplift-v2.1.csv",
    "--frac",
    "0.5",
    "--split",
    "sprint3",
    "--profile",
    "rare-outcome",
  

/kaggle/working/repo/causal_forest_rare_outcome.zip

## Cell 11 — Kết luận

Đặt **sau** bước đóng gói, để một lần gate fail không làm mất artifact vừa fit xong.

Hai loại thất bại phải phân biệt rõ:

| | Ý nghĩa | Điểm số dùng được? |
|---|---|---|
| `all_finite` hoặc `aligned` = False | Artifact hỏng | **Không** |
| Chỉ vượt ngưỡng RAM | Fit xong, vi phạm ngân sách tài nguyên đã đăng ký | **Có**, nhưng phải ghi rõ mức vượt |

In [10]:
contract = manifest['artifact_contract']
runtime = manifest['runtime']
fraction = runtime['peak_process_tree_ram_fraction']
limit = manifest['stop_rule']['max_ram_fraction']

artifact_ok = bool(contract['all_finite'] and contract['aligned']
                   and contract['score_rows'] == 1_397_959)
ram_ok = fraction < limit

print('artifact hop le      ', artifact_ok)
print(f'RAM trong ngan sach   {ram_ok}  ({fraction:.3f} so voi nguong {limit})')
print()

assert artifact_ok, (
    'Artifact HONG - khong dung diem so nay. '
    f"score_rows={contract['score_rows']} finite={contract['all_finite']} "
    f"aligned={contract['aligned']}"
)

if ram_ok:
    print('Gate PASS. Tai zip ve va cham diem o local.')
else:
    print(f'Gate FAIL vi RAM: peak {fraction:.1%} vuot nguong {limit:.0%}.')
    print()
    print('Diem so VAN HOP LE - fit da chay xong, artifact dung so dong va aligned.')
    print('Hai lua chon:')
    print('  1. Dung ket qua nay, ghi ro muc vuot RAM trong bao cao.')
    print('  2. Chay lai voi N_JOBS = 2 o Cell 9 de qua gate.')
    print('     Da kiem chung n_jobs KHONG doi diem so, nen day khong phai tinh chinh')
    print('     sau khi nhin ket qua - chi la doi cach chay.')

artifact hop le       True
RAM trong ngan sach   False  (0.906 so voi nguong 0.75)

Gate FAIL vi RAM: peak 90.6% vuot nguong 75%.

Diem so VAN HOP LE - fit da chay xong, artifact dung so dong va aligned.
Hai lua chon:
  1. Dung ket qua nay, ghi ro muc vuot RAM trong bao cao.
  2. Chay lai voi N_JOBS = 2 o Cell 9 de qua gate.
     Da kiem chung n_jobs KHONG doi diem so, nen day khong phai tinh chinh
     sau khi nhin ket qua - chi la doi cach chay.


## Bước tiếp theo — chấm điểm ở local

Kaggle chỉ fit và xuất điểm CATE. Việc chấm phải chạy ở máy có
`output/sprint3/confirmation_predictions.npz` (file bị `.gitignore` loại nên không có trong bản
clone).

Giải nén `causal_forest_rare_outcome.zip` vào `output/causal_forest/`, rồi:

```powershell
.venv\Scripts\python.exe scripts\evaluate_causal_forest.py `
  --stage-dir output\causal_forest\sprint3_rare_outcome `
  --score-name cate_causal_forest_rare_outcome.npy `
  --n-boot 500
```

Script sẽ:

1. đối chiếu `source_index` **trùng khít từng phần tử** với bảng confirmation Sprint 3 — lệch
   một dòng là dừng, không chấm;
2. dùng đúng **DR signal đã đóng băng** của Sprint 3, không fit lại nuisance;
3. so Causal Forest với cả chín model bằng paired bootstrap dùng chung bootstrap weights;
4. ghi `cf_sprint3_metrics.csv`, `cf_sprint3_paired_comparisons.csv` và
   `cf_sprint3_summary.json` vào `output/causal_forest/release/`.

Đọc `policy_area_ci_low` ở dòng `model_b = Response`: chỉ khi nó **lớn hơn 0** thì mới có bằng
chứng Causal Forest vượt champion. Mọi kết quả khác là hoà hoặc thua, và champion giữ nguyên.